In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from workflow.scripts.utils import global_avg,read_list_input_paths
from workflow.scripts.utils import calc_error, calc_error_gridded, regrid_global
from workflow.scripts.plotting_tools import create_facet_plot, get_model_colordict
import numpy as np

In [ ]:
lon_slice = slice(-60,36)
lat_slice = slice(5,31)

In [ ]:
order = [             
            'GFDL-ESM4',
            'CNRM-ESM2-1',
            'UKESM1-0-LL',
            'IPSL-CM6A-LR-INCA',
            'NorESM2-LM',
            'MPI-ESM-1-2-HAM',
            'EC-Earth3-AerChem'            
        ]
cdict = get_model_colordict()

In [ ]:
rsds = read_list_input_paths(snakemake.input.rsds)[0]
rsus = read_list_input_paths(snakemake.input.rsus)[0]

rsds = {m: rsds[m].isel(time=slice(1,None)).mean(dim='time') for m in rsds}
rsus = {m: rsus[m].isel(time=slice(1,None)).mean(dim='time') for m in rsus}
def calc_albedo(rsds, rsus):
    return rsus['rsus']/rsds['rsds']

surfal = {m :  calc_albedo(rsds[m], rsus[m]).sel(lon=lon_slice, lat=lat_slice) for m in rsds}

In [ ]:
ds_exp = {p.split("_")[-2]: xr.open_dataset(p).isel(time=slice(1,None)).sel(lon=lon_slice, lat=lat_slice) for p in snakemake.input.exp_data}
ds_ctrl = {p.split("_")[-2]: xr.open_dataset(p).sel(lon=lon_slice, lat=lat_slice) for p in snakemake.input.ctrl_data}
diff = {k:ds_exp[k].mean(dim='time') - ds_ctrl[k].mean(dim='time') for k in ds_ctrl}

In [ ]:
paths = sorted(snakemake.input.paths)
vname = snakemake.wildcards.vName

params = snakemake.params
time_slice = params.get('time_slice', slice(3,-1))
nlevels = params.get('nlevels', 11)
draw_error_mask = params.get('draw_error_mask', True)

In [ ]:
dsets = {}
for path in paths:

    ds = xr.open_dataset(path)
    source_id = ds.source_id
        
    dsets[source_id]={}
    ds = ds[snakemake.wildcards.vName].isel(year=time_slice)
    ds = ds.mean(dim='year',keep_attrs=True)
    # percentile = np.percentile(np.abs(ds),75)
    # mask = np.abs(ds) < percentile
    # ds = ds.where(~mask)
        
    dsets[source_id]=ds.sel(lon=lon_slice, lat=lat_slice)

     
    

In [ ]:
unit_forcing = "[$\mathrm{W\;m}^{-2}$]"

In [ ]:
ylabel_translator = {
    'DirectEff': 'Dust direct forcing efficiency',
    'SWDirectEff' : 'Dust SW direct forcing efficiency',
    'LWDirectEff' : 'Dust LW direct forcing efficiency',
}

In [ ]:
fig, (ax,ax1) = plt.subplots(ncols=2, figsize=(8.7,3.6), sharex=True)
for m in order[::-1]:
    surfal[m].mean(dim='lat').plot(ax=ax,label=m, color=cdict[m])
    od = diff[m]['od550dust'].mean(dim='lat')
    (dsets[m].mean(dim='lat').interp_like(od)/od).plot(ax=ax1,label=m, color=cdict[m])
    ax1.set_title('')
    ax1.set_xlabel('Longitude')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Surface Albedo')
    ax1.set_ylabel(ylabel_translator.get(snakemake.wildcards.vName,snakemake.wildcards.vName)+f' {unit_forcing}')
    ax.text(0.05,.93,'a)', transform=ax.transAxes)
    ax1.text(0.05,.93,'b)', transform=ax1.transAxes)
    if snakemake.wildcards.vName == 'LWDirectEff':
        ax1.set_ylim(-5,30)
    else:
        ax1.set_ylim(-45,20)
h,l = ax.get_legend_handles_labels()
fig.legend(h,l,ncols=3,loc='lower center',bbox_to_anchor=(0.5,-0.22))
fig.subplots_adjust(wspace=0.3)
plt.savefig(snakemake.output.outpath, bbox_inches='tight', dpi=300)